# CLV 이중축 M2 seed 43·44 validation 재현성

기존 seed 42 결과를 재사용하고 Dunnhumby 전체기간과 H&M 60일에서 seed 43·44의 M1 및 고정 `dual_clv_fixed`만 실행합니다. test·holdout·대조군·H&M 전체기간은 실행하지 않으며, 성공해도 후속 실험을 자동 시작하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '69acf846aeb0dc4e75c77cb8359f8525867a1f72'
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import torch
from lightgcn_clv_dual_multiseed import (
    configure_multiseed_validation,
    run_multiseed_validation,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
DRIVE_ROOT = Path('/content/drive/MyDrive/논문/data')
DUN_FOLDER = DRIVE_ROOT / 'results_clv_dual_dunnhumby'
HM_FOLDER = DRIVE_ROOT / 'results_clv_dual_hm_w60'

def latest_seed42_json(folder):
    candidates = [path for path in folder.glob('clv_dual_*.json') if 'multiseed' not in path.name]
    if not candidates:
        raise FileNotFoundError(f'seed 42 dual 결과 JSON이 없습니다: {folder}')
    return max(candidates, key=lambda path: path.stat().st_mtime)

configs = {
    'dunnhumby': configure_multiseed_validation(
        'dunnhumby',
        latest_seed42_json(DUN_FOLDER),
        out_dir=DUN_FOLDER / 'multiseed_validation',
    ),
    'hm_w60': configure_multiseed_validation(
        'hm',
        latest_seed42_json(HM_FOLDER),
        short_hm=True,
        out_dir=HM_FOLDER / 'multiseed_validation',
    ),
}
for name, cfg in configs.items():
    print(name, cfg)


## 실행

아래 셀은 seed 43·44에서 데이터셋별 encoder, 같은 seed M1, `dual_clv_fixed`를 학습합니다. λ sweep은 하지 않고 Dunnhumby `equal, λ=2.0`, H&M 60일 `high, λ=1.0` 한 점만 평가합니다.

In [ ]:
results = {}
for name in ('dunnhumby', 'hm_w60'):
    print(f'\n===== {name}: seed 43·44 validation 시작 =====')
    results[name] = run_multiseed_validation(configs[name])
    print(f'===== {name}: 완료 =====')


In [ ]:
for name, frame in results.items():
    print(f'\n===== {name}: 3-seed 절대지표 =====')
    display(frame[[
        'seed', 'model_id', 'gate_shape', 'lambda',
        'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50', 'revenue@10', 'arp@10',
        'coverage@10', 'n_distinct@10', 'eff_catalog@10',
        'top10_share@10', 'top100_share@10',
    ]].sort_values(['seed', 'model_id']))
    reproducibility_decision = frame.attrs['reproducibility_decision']
    print('재현성 통과:', reproducibility_decision['success'])
    print('실패 조건:', reproducibility_decision['failed_conditions'])
    print('seed별 경제지표 delta:', reproducibility_decision['seed_revenue_delta'])
    print('3-seed 평균 경제지표 delta:', reproducibility_decision['mean_revenue_delta'])
    print('정확도 평균 비율:', reproducibility_decision['accuracy_mean_ratios'])
    print('결과 파일:', frame.attrs['result_paths'])

print('완료: 여기서 중단합니다. 결과를 검토하기 전에는 다음 고비용 실험을 실행하지 않습니다.')
